In [2]:
import sqlalchemy
import mlflow

print("SQLAlchemy:", sqlalchemy.__version__)
print("MLflow:", mlflow.__version__)

SQLAlchemy: 2.1.9
MLflow: 3.1.1


In [3]:
%pip install sqlalchemy==2.0.44

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 8.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
  Attempting uninstall: sqlalchemy
   ---------------------------------------- 0/2 [greenlet]
    Found existing installation: SQLAlchemy 2.1.9
   ---------------------------------------- 0/2 [greenlet]
   -------------------- ------------------- 1/2 [sqlalchemy]
    Uninstalling SQLAlchemy-2.1.9:
   -------------------- ------------------- 1/2 [sqlalchemy]
      Successfully uninstalled SQLAlchemy-2.1.9
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   ------------

  You can safely remove it manually.


In [1]:
import sqlalchemy
import mlflow

print("SQLAlchemy:", sqlalchemy.__version__)
print("MLflow:", mlflow.__version__)

print("MLflow imported successfully!")

SQLAlchemy: 2.0.44
MLflow: 3.1.1
MLflow imported successfully!


In [2]:
import mlflow
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

mlflow.set_tracking_uri(
    f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"
)

EXPERIMENT_NAME = "MP002_Canteen_Demand_Forecasting"

mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

2026/11/21 20:51:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/21 20:51:08 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

Tracking URI: sqlite:///d:/College_Canteen_lunchRush/College_Canteen_lunchRush/mlflow.db
Experiment: MP002_Canteen_Demand_Forecasting


In [3]:
from pathlib import Path
import pandas as pd
import joblib

PROJECT_ROOT = Path.cwd().parent

v1_model = joblib.load(PROJECT_ROOT / "models" / "v1_model.pkl")

train_df = pd.read_csv(PROJECT_ROOT / "data/processed/train_clean.csv")
later_df = pd.read_csv(PROJECT_ROOT / "data/raw/later.csv")

FEATURES = [
    "students_on_campus",
    "menu_price_rs",
    "temp_c",
    "exam_days_left",
    "special_dishes",
    "queue_minutes_yesterday",
    "day_type",
    "block"
]

X = train_df[FEATURES]
X_later = later_df[FEATURES]
y_later = later_df["plates_sold"]

# Apply the same unit correction used during training
mask = X_later["block"] == "B-Block"
X_later.loc[mask, "menu_price_rs"] /= 100

print("V1 model loaded successfully!")

V1 model loaded successfully!


C:\Users\riyag\AppData\Local\Temp\ipykernel_43788\3885133528.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.32 0.65 0.46 0.63 0.41 0.87 0.64 0.75 0.79 0.32 0.63 1.18 0.75 0.69
 1.05 0.38 1.   1.1  0.73 0.57 1.18 0.68 0.63 0.75 0.83 1.05 1.14 0.85
 0.7  0.97 0.31 1.09 0.76 0.59 0.76 0.59 0.48 0.36 0.55 1.11 1.06 0.7
 0.85 0.57 0.34 0.95 0.53 1.02 0.51 0.41 0.64 0.35 0.52 0.54 0.44 0.91
 0.8  0.58 0.77 0.39 0.88 0.53 0.41 0.85 0.63]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_later.loc[mask, "menu_price_rs"] /= 100


In [4]:
from sklearn.metrics import mean_absolute_error

predictions = v1_model.predict(X_later)

v1_later_mae = mean_absolute_error(y_later, predictions)

print("V1 MAE on later dataset:", round(v1_later_mae, 2))

V1 MAE on later dataset: 29.88


In [5]:
import hashlib

train_path = PROJECT_ROOT / "data/raw/train.csv"

with open(train_path, "rb") as f:
    data_sha256 = hashlib.sha256(f.read()).hexdigest()

print("SHA256:", data_sha256)

SHA256: 3adf21774755bb6a10dfd0bcc4860839e64573a7f31eb065d491eff1a1b90f5f


In [6]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

mlflow.set_tracking_uri(
    f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"
)

mlflow.set_experiment("MP002_Canteen_Demand_Forecasting")

input_example = X.head(5)
output_example = v1_model.predict(input_example)

signature = infer_signature(input_example, output_example)

with mlflow.start_run(run_name="Linear_Regression_V1") as run:

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features_count", len(FEATURES))
    mlflow.log_param("training_rows", len(train_df))
    mlflow.log_param("data_sha256", data_sha256)

    mlflow.log_metric("validation_mae", 13.68)
    mlflow.log_metric("later_mae", v1_later_mae)

    mlflow.sklearn.log_model(
        sk_model=v1_model,
        name="model",
        signature=signature,
        input_example=input_example,
        registered_model_name="canteen_demand_model"
    )

    v1_run_id = run.info.run_id

print("V1 Run ID:", v1_run_id)
print("V1 registered successfully!")

d:\College_Canteen_lunchRush\College_Canteen_lunchRush\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/11/21 20:52:52 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
2026/11/21 20:52:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/21 20:52:52 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


V1 Run ID: 741a40ff7b8c42d6a0467931bea30515
V1 registered successfully!


Successfully registered model 'canteen_demand_model'.
Created version '1' of model 'canteen_demand_model'.


In [7]:
client = mlflow.MlflowClient()

for version in client.search_model_versions(
    "name='canteen_demand_model'"
):
    print("Version:", version.version)
    print("Run ID:", version.run_id)

Version: 1
Run ID: 741a40ff7b8c42d6a0467931bea30515
